<a href="https://colab.research.google.com/github/iaprojectsit/M08-Tecnicas-de-Construcao-de-Algoritmos-Quanticos/blob/main/UA2_Computa%C3%A7%C3%A3o_Adiab%C3%A1tica_(D_Wave_Ocean_SDK).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**# Bloco 3: Infraestrutura e Grover Adiabático (Problema de Busca)**

In [1]:
# Commit: [env] Prepara o simulador termodinâmico e otimização de grafos
!pip install -q dwave-ocean-sdk dimod networkx dwave-networkx

import dimod

# Commit: [logic] Molde QUBO para Grover: Estado (q0=1, q1=0) deve ter a menor energia
pesos_lineares = {'q0': -1.0, 'q1': 1.0}
bqm_grover = dimod.BinaryQuadraticModel(pesos_lineares, {}, offset=0.0, vartype=dimod.BINARY)

# Commit: [test] Inicia o recozimento clássico (Annealing)
sampler = dimod.SimulatedAnnealingSampler()
resposta_grover = sampler.sample(bqm_grover, num_reads=10)

print("--- Grover Adiabático (A Bacia de Energia) ---")
print(resposta_grover)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
--- Grover Adiabático (A Bacia de Energia) ---
  q0 q1 energy num_oc.
1  1  0   -1.0       1
5  1  0   -1.0       1
6  1  0   -1.0       1


**# Bloco 4: Análise Termodinâmica (Prova de Ruído para o Relatório)**

In [2]:
# Commit: [test] Cenário 1: Isolamento Térmico Perfeito (Beta Alto)
print("--- Cenário 1: Frio Extremo (100% de Precisão) ---")
resp_frio = sampler.sample(bqm_grover, num_reads=100, beta_schedule_type='linear', beta_range=(0.1, 10.0))
print(resp_frio.aggregate())

# Commit: [test] Cenário 2: Colapso por Ruído Térmico (Beta Baixo = Temperatura Alta)
print("\n--- Cenário 2: Terremoto Térmico (Falha na Convergência) ---")
resp_quente = sampler.sample(bqm_grover, num_reads=100, beta_schedule_type='linear', beta_range=(0.01, 0.1))
print(resp_quente.aggregate())

--- Cenário 1: Frio Extremo (100% de Precisão) ---
  q0 q1 energy num_oc.
0  1  0   -1.0     100
['BINARY', 1 rows, 100 samples, 2 variables]

--- Cenário 2: Terremoto Térmico (Falha na Convergência) ---


/tmp/ipykernel_814/241095889.py:3: SamplerUnknownArgWarning: Ignoring unknown kwarg: 'beta_schedule_type'
  resp_frio = sampler.sample(bqm_grover, num_reads=100, beta_schedule_type='linear', beta_range=(0.1, 10.0))
/tmp/ipykernel_814/241095889.py:8: SamplerUnknownArgWarning: Ignoring unknown kwarg: 'beta_schedule_type'
  resp_quente = sampler.sample(bqm_grover, num_reads=100, beta_schedule_type='linear', beta_range=(0.01, 0.1))


  q0 q1 energy num_oc.
3  1  0   -1.0      27
0  0  0    0.0      21
2  1  1    0.0      27
1  0  1    1.0      25
['BINARY', 4 rows, 100 samples, 2 variables]


**# Bloco 5: O Problema do Caixeiro Viajante (TSP)**

In [3]:
import networkx as nx
import dwave_networkx as dnx

# Commit: [setup] Constrói a topografia geométrica das cidades e distâncias
G = nx.Graph()
G.add_weighted_edges_from([
    (0, 1, 360), # Distância Cidade 0 a 1
    (1, 2, 490), # Distância Cidade 1 a 2
    (2, 0, 340)  # Distância Cidade 2 a 0
])

# Commit: [logic] Executa a abstração matemática de Penalidades vs. Custo (O Combustível)
rota_otima = dnx.traveling_salesperson(G, sampler, start=0)

print(f"\nRota de menor energia calculada pelo modelo adiabático: {rota_otima}")


Rota de menor energia calculada pelo modelo adiabático: [0, 1, 2]
